In [14]:
import os
import glob
import os
from scipy.io import mmwrite
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml
import time
import re
#import gget
import psutil
from scipy.stats import zscore

import rmm
import cudf
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator
import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

sc.settings.verbosity = 3

In [15]:
# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

# Reference Atlas

In [27]:
%%time
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hematokytos/processed/sample_1_adata.h5ad"
adata = sc.read_h5ad(fpath)
adata

CPU times: user 12.1 s, sys: 52.7 s, total: 1min 4s
Wall time: 5min 2s


/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.13/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 1125041 × 52164
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'X_umap_X_scANVI', 'X_umap_raw_data', 'X_umap_scVI', '_scvi_manager_uuid', '_scvi_uuid', 'basename_colors', 'basename_palette

In [ ]:
rsc.get.anndata_to_GPU(adata) # move to GPU
rsc.pp.filter_genes(adata, min_counts=500)
adata

filtered out 23626 genes that are detected in less than 500 counts


In [ ]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/basename_ref.h5ad"

aggdata = sc.get.aggregate(
    adata,
    by='basename',
    func='sum',
    layer='counts'
)

aggdata.X = aggdata.layers['sum']
del aggdata.layers['sum']
print(f"{aggdata.shape=}")
#aggdata.write(outpath)
aggdata.obs.head()


In [ ]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/cell_type_ref.h5ad"

aggdata = sc.get.aggregate(
    adata,
    by='cell_type',
    func='sum',
    layer='counts',
)

aggdata.X = aggdata.layers['sum']
del aggdata.layers['sum']
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()

In [ ]:
counts = adata.obs['cell_type'].value_counts().reset_index()
counts = counts.sort_values('cell_type')
print(counts.to_string(index=False))

# Hand-selected cell types + downsampling

In [10]:
%%time
cell_types = [
    'embryonic fibroblast',
    'skin fibroblast',
    # 'fibro/adipogenic progenitor cell',
    'hematopoietic stem cell',
    # 'cord blood hematopoietic stem cell',
    'common myeloid progenitor',
    'early lymphoid progenitor',
    'megakaryocyte-erythroid progenitor cell',
    'granulocyte monocyte progenitor cell',
    'hematopoietic multipotent progenitor cell',
    'endothelial cell of artery',
    'endothelial cell of lymphatic vessel',
    'vein endothelial cell',
]
sample_size = 5000

print("[INFO] Filtering AnnData for selected cell types...")
bdata = adata[adata.obs['cell_type'].isin(cell_types)].copy()
print(f"[INFO] Shape after cell type filtering: {bdata.shape}")

# Remove duplicate obs_names
print("[INFO] Removing duplicate obs_names...")
duplicates = bdata.obs_names.duplicated(keep='first')
n_duplicates = duplicates.sum()
if n_duplicates > 0:
    print(f"[INFO] Found {n_duplicates} duplicate cells. Removing.")
    bdata = bdata[~duplicates].copy()
else:
    print("[INFO] No duplicate cells found.")
print(f"[INFO] Shape after duplicate removal: {bdata.shape}")

print("[INFO] Cell counts per type (before downsampling):")
print(bdata.obs['cell_type'].value_counts().to_string())
print("="*60)

# Downsample cells per type
print(f"[INFO] Sampling up to {sample_size} cells per cell type...")
grouped = bdata.obs.groupby('cell_type', observed=True)
sampled_indices = []
for cell_type, group in grouped:
    n = min(sample_size, len(group))
    sampled = group.sample(n=n, random_state=0).index.tolist()
    sampled_indices.extend(sampled)

# Now sampled_indices is a flat list of obs_names
bdata = bdata[sampled_indices, :].copy()
print(f"[INFO] Shape after downsampling: {bdata.shape}")

min_counts = 25
print(f"[INFO] Filtering genes based on min counts: {min_counts}")
rsc.pp.filter_genes(bdata, min_counts = min_counts)
print(f"[INFO] Filtered shape: {bdata.shape}")

print("\n[INFO] Cell counts per type (after downsampling):")
print(bdata.obs['cell_type'].value_counts().to_string())
print("="*60)

[INFO] Filtering AnnData for selected cell types...
[INFO] Shape after cell type filtering: (116913, 28538)
[INFO] Removing duplicate obs_names...
[INFO] Found 6920 duplicate cells. Removing.
[INFO] Shape after duplicate removal: (109993, 28538)
[INFO] Cell counts per type (before downsampling):
cell_type
hematopoietic multipotent progenitor cell    19820
vein endothelial cell                        18914
skin fibroblast                              17272
endothelial cell of lymphatic vessel         14759
hematopoietic stem cell                      12603
endothelial cell of artery                    8854
megakaryocyte-erythroid progenitor cell       7169
granulocyte monocyte progenitor cell          3608
early lymphoid progenitor                     2690
common myeloid progenitor                     2595
embryonic fibroblast                          1709
[INFO] Sampling up to 5000 cells per cell type...
[INFO] Shape after downsampling: (45602, 28538)
[INFO] Filtering genes based on mi

# Aggregation (pseudobulk)

In [11]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/hand_select.h5ad"

agg = 'sum'

aggdata = sc.get.aggregate(
    bdata,
    by='cell_type',
    func=agg,
    layer='counts',
)

aggdata.X = aggdata.layers[agg]
del aggdata.layers[agg]
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()

aggdata.shape=(11, 23858)
CPU times: user 452 ms, sys: 438 ms, total: 890 ms
Wall time: 989 ms


,cell_type
common myeloid progenitor,common myeloid progenitor
early lymphoid progenitor,early lymphoid progenitor
embryonic fibroblast,embryonic fibroblast
endothelial cell of artery,endothelial cell of artery
endothelial cell of lymphatic vessel,endothelial cell of lymphatic vessel


# Ng Data

In [4]:
%%time
fpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/new_processed/Ng_2024_raw.h5ad"
adata = sc.read_h5ad(fpath)
adata

CPU times: user 3.68 s, sys: 12.8 s, total: 16.5 s
Wall time: 18.3 s


AnnData object with n_obs × n_vars = 252607 × 27946
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'induction.ident', 'celltype', 'RETA'
    var: 'name'
    obsm: 'X_pca', 'X_umap'
    layers: 'raw_counts'

In [5]:
rsc.get.anndata_to_GPU(adata) # move to GPU
rsc.pp.filter_genes(adata, min_counts=500)
adata

filtered out 11503 genes that are detected in less than 500 counts


AnnData object with n_obs × n_vars = 252607 × 16443
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'induction.ident', 'celltype', 'RETA'
    var: 'name', 'n_counts', 'n_cells'
    obsm: 'X_pca', 'X_umap'
    layers: 'raw_counts'

In [6]:
adata.obs["celltype"].unique()

['HSPC2', 'EMK Pro1', 'HSPC1', 'Hemogenic', 'Mye Pro2', ..., 'Endo', 'Strom1', 'Strom3', 'Venous', 'Art2']
Length: 20
Categories (20, object): ['Strom1', 'Strom2', 'Strom3', 'Strom4', ..., 'Mye Pro1', 'Mye Pro2', 'Mono', 'Gran']

In [7]:
orig_to_capy = {
    "HSPC1": "HSPC1",
    "HSPC2": "HSPC2",
    "HSPC3": "HSPC3",
    "Strom1": "Stroma",
    "Strom2": "Stroma",
    "Strom3": "Stroma",
    "Strom4": "Stroma",
    "Art1": "Vessel_endothelium",
    "Art2": "Vessel_endothelium",
    "Venous": "Vessel_endothelium",
    "Gran": "Gran/Mono",
    "Mono": "Gran/Mono",
    "Eos": "Gran/Mono",
    "Mega": "Erythroid/Mega",
    "EMK Pro1": "Erythroid/Mega",
    "EMK Pro2": "Erythroid/Mega",
    "Mye Pro1": "Myeloid",
    "Mye Pro2": "Myeloid",
    "Hemogenic": "Hemogenic_endothelium",
    "Endo": "Endothelial",
}

# overwrite celltype (or write to a new column instead)
adata.obs["celltype"] = adata.obs["celltype"].map(orig_to_capy).fillna(adata.obs["celltype"])

In [9]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/ng_ref.h5ad"

aggdata = sc.get.aggregate(
    adata,
    by='celltype',
    func='sum',
    layer='raw_counts',
)

aggdata.X = aggdata.layers['sum']
del aggdata.layers['sum']
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()


aggdata.shape=(10, 16443)
CPU times: user 45.8 s, sys: 13.7 s, total: 59.5 s
Wall time: 60 s


,celltype
Endothelial,Endothelial
Erythroid/Mega,Erythroid/Mega
Gran/Mono,Gran/Mono
HSPC1,HSPC1
HSPC2,HSPC2


In [10]:
counts = adata.obs['celltype'].value_counts().reset_index()
counts = counts.sort_values('celltype')
print(counts.to_string(index=False))

             celltype  count
          Endothelial  12547
       Erythroid/Mega  45009
            Gran/Mono  35854
                HSPC1  35744
                HSPC2  10662
                HSPC3   6559
Hemogenic_endothelium   9108
              Myeloid  41368
               Stroma  36406
   Vessel_endothelium  19350


# Hand-selected cell types + downsampling

In [11]:
%%time
cell_types = [
    'HSPC2', 'Erythroid/Mega', 'HSPC1', 'Hemogenic_endothelium',
    'Myeloid', 'Gran/Mono', 'HSPC3', 'Vessel_endothelium', 'Stroma',
    'Endothelial'
]
sample_size = 5000

print("[INFO] Filtering AnnData for selected cell types...")
bdata = adata[adata.obs['celltype'].isin(cell_types)].copy()
print(f"[INFO] Shape after cell type filtering: {bdata.shape}")

# Remove duplicate obs_names
print("[INFO] Removing duplicate obs_names...")
duplicates = bdata.obs_names.duplicated(keep='first')
n_duplicates = duplicates.sum()
if n_duplicates > 0:
    print(f"[INFO] Found {n_duplicates} duplicate cells. Removing.")
    bdata = bdata[~duplicates].copy()
else:
    print("[INFO] No duplicate cells found.")
print(f"[INFO] Shape after duplicate removal: {bdata.shape}")

print("[INFO] Cell counts per type (before downsampling):")
print(bdata.obs['celltype'].value_counts().to_string())
print("="*60)

# Downsample cells per type
print(f"[INFO] Sampling up to {sample_size} cells per cell type...")
grouped = bdata.obs.groupby('celltype', observed=True)
sampled_indices = []
for cell_type, group in grouped:
    n = min(sample_size, len(group))
    sampled = group.sample(n=n, random_state=0).index.tolist()
    sampled_indices.extend(sampled)

# Now sampled_indices is a flat list of obs_names
bdata = bdata[sampled_indices, :].copy()
print(f"[INFO] Shape after downsampling: {bdata.shape}")

min_counts = 25
print(f"[INFO] Filtering genes based on min counts: {min_counts}")
rsc.pp.filter_genes(bdata, min_counts = min_counts)
print(f"[INFO] Filtered shape: {bdata.shape}")

# n_hvg = 1000
# print(f"[INFO] Filtering {n_hvg} HVG...")
# rsc.pp.highly_variable_genes(bdata, n_top_genes=n_hvg, batch_key='cell_type')
# bdata = bdata[:, bdata.var['highly_variable']].copy()
# print(f"[INFO] Filtered shape: {bdata.shape}")

print("\n[INFO] Cell counts per type (after downsampling):")
print(bdata.obs['celltype'].value_counts().to_string())
print("="*60)

[INFO] Filtering AnnData for selected cell types...
[INFO] Shape after cell type filtering: (252607, 16443)
[INFO] Removing duplicate obs_names...
[INFO] No duplicate cells found.
[INFO] Shape after duplicate removal: (252607, 16443)
[INFO] Cell counts per type (before downsampling):
celltype
Erythroid/Mega           45009
Myeloid                  41368
Stroma                   36406
Gran/Mono                35854
HSPC1                    35744
Vessel_endothelium       19350
Endothelial              12547
HSPC2                    10662
Hemogenic_endothelium     9108
HSPC3                     6559
[INFO] Sampling up to 5000 cells per cell type...
[INFO] Shape after downsampling: (50000, 16443)
[INFO] Filtering genes based on min counts: 25
[INFO] Filtered shape: (50000, 16443)

[INFO] Cell counts per type (after downsampling):
celltype
Endothelial              5000
Erythroid/Mega           5000
Gran/Mono                5000
HSPC1                    5000
HSPC2                    5000
HSP

# Aggregation (pseudobulk)

In [12]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/ng_ref.h5ad"

agg = 'sum'

aggdata = sc.get.aggregate(
    bdata,
    by='celltype',
    func=agg,
    layer='raw_counts',
)

aggdata.X = aggdata.layers[agg]
del aggdata.layers[agg]
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()

aggdata.shape=(10, 16443)
CPU times: user 7.28 s, sys: 2.27 s, total: 9.55 s
Wall time: 9.66 s


,celltype
Endothelial,Endothelial
Erythroid/Mega,Erythroid/Mega
Gran/Mono,Gran/Mono
HSPC1,HSPC1
HSPC2,HSPC2


# Gomes

In [4]:
%%time
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/data/datasets/gomes_2018/gomes.h5ad"
adata = sc.read_h5ad(fpath)
adata

CPU times: user 65.6 ms, sys: 63.7 ms, total: 129 ms
Wall time: 892 ms


/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.13/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 286 × 56269
    obs: 'cell_type', 'cell_num'
    layers: 'raw_counts'

In [ ]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/gomes_ref.h5ad"

aggdata = sc.get.aggregate(
    adata,
    by='cell_type',
    func='sum',
)

aggdata.X = aggdata.layers['sum']
del aggdata.layers['sum']
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()

In [ ]:
aggdata